In [2]:
%pip uninstall chromadb -y

Found existing installation: chromadb 1.5.9
Uninstalling chromadb-1.5.9:
  Successfully uninstalled chromadb-1.5.9
Note: you may need to restart the kernel to use updated packages.


In [10]:
#--------------langchain text chunker--------------------
#!pip install --upgrade uuid-utils
#!pip install -U langchain-text-splitters

In [ ]:
#------langchain community for reading html--------------------
#%conda install -c conda-forge langchain-community beautifulsoup4

In [5]:
#---------------------install chromadb---------------------------
#https://anaconda.org/channels/conda-forge/packages/chromadb/overview
#%conda install conda-forge::chromadb

In [13]:
import os
import argparse
from gpt4all import GPT4All, Embed4All
import chromadb
import pymupdf4llm
from pypdf import PdfReader  # Added for PDF support
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import BSHTMLLoader

In [ ]:
#pip install pydantic==2.6.4 pydantic-core==2.46.4 --target=./.chroma_deps

In [7]:
#%conda install conda-forge::langchain-text-splitters
#%conda install langchain

In [8]:
#!pip install pypdf
#!pip install pymupdf4llm

In [6]:
embed_params={
    "llm" : GPT4All("Llama-3.2-3B-Instruct-Q4_0.gguf",device="gpu"),
"embedder" : Embed4All(),
"chroma_client" : chromadb.PersistentClient(path="./chroma_db")}

In [7]:
#---------------create a list of files by format (extension) in a given path
def list_files(path):
    files= {'pdf':[],'txt':[],"html":[]}
    for filename in os.listdir(path):
        if filename.endswith(".txt"):
            files["txt"].append( os.path.join(path, filename) )
        elif filename.endswith(".pdf"):
            files["pdf"].append( os.path.join(path, filename) )
        elif filename.endswith(".html"):
            files["html"].append( os.path.join(path, filename) )
    return files

files=list_files("{}/gpt4alldocs/test2".format("path"))            

In [ ]:
#--------------------extract text from a single file according to its extension
def extract_text_from_file(file_path,extension="pdf"):
    print('file path: ', file_path)
    # Extracts the entire layout natively into clean Markdown chunks
    if extension=="pdf": md_text = pymupdf4llm.to_markdown(file_path)
    #paragraphs = [p.strip() for p in md_text.split("\n\n") if p.strip()]
    #https://www.educative.io/answers/what-is-bshtmlloader-in-langchain
    if extension=="html": #NOTE THIS IS NOT PARSED AS MARKDOWN BUT PLAIN TXT
        # Explicitly pass the 'html.parser' as the parser
        loader = BSHTMLLoader(file_path, bs_kwargs={'features': 'html.parser'})
        # Load the document (this will parse and extract the content from the HTML file)
        md_text = loader.load()[0]
        md_text=md_text.page_content
    
    
    return md_text

#text=extract_text_from_file(file,extension="html") #unit test

In [ ]:
#----------------extract text from a set of files, applying the extract text in a single file function
def extract_text_from_files(files):
    texts=[]
    for file in files["pdf"]:
        print('extracting text from file: {}'.format(file))
        texts.append({"file":file,"text":extract_text_from_file(file,extension="pdf")})
    for file in files["html"]:
        print('extracting text from file: {}'.format(file))
        texts.append({"file":file,"text":extract_text_from_file(file,extension="html")})
        
    return texts

texts=extract_text_from_files(files)

#### **Chunk Texts**
- For storing the data in the vector database (ChromaDB), each document has to be split into chunks.
- This is necessary later for RAG, as the system first retrieves relevant chunks across all documents, and adds them to the prompt.
- There are several methods for chunking:
    - In the example we use recursive character chunking, for dynamically detecting relevant structure in text such as paragraphs.
- Langchain Docs:
    - https://docs.langchain.com/oss/python/integrations/splitters
    - https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter
- **Note**: In ChromaDB "documents" refer to the "chunks" in this case (the atomic entity). We are storing all the chunks across all the source documents (in this case files), as the RAG engine will search through all of them in a "flat" structure. For associating every chunk with its source file we can use the metadata.

In [ ]:
# Langchain Parameters
# Typical paragraphs have between 300-600 characters (chunk_size param)
langchain_params={"chunk_size":600, "chunk_overlap":0,"add_start_index":True}

def chunk_text(text,filename=None,langchain_params={"chunk_size":600, "chunk_overlap":0,"add_start_index":True}):
    text_splitter = RecursiveCharacterTextSplitter(**langchain_params)
    print('--------------------chunking text')
    texts = text_splitter.split_text(text) #split text in chunks (1 "text"-> 1 chunk)
    chunk_id=0
    chunks=[] #container for chunks with metadata
    print("----------------adding metadata-----------")
    if isinstance(filename,str): "print file {}".format(filename)
    for c_idx,chunk  in enumerate(texts):
        # Ensure the paragraph isn't completely empty (not necessary with langchain text splitter)
        #if len(chunk) > 10:  
        chunks.append( {"documents":[chunk],
                        "metadatas":[{
                            "source_file": filename, 
                            #"page": page_num,
                            "f_chunk_idx": c_idx
                        }],
                        "ids":[f"chunk_{chunk_id}"]
           }
                    )
        chunk_id += 1
    
    return chunks
    
#flatten list of chunks
#Instead of nesting chunks by document, we create a single list of chunks accross al documents, for ingesting into ChromaDB
chunks=[chunk_text(text["text"],filename=text["file"],langchain_params=langchain_params) for text in texts ]
chunks = [item for sublist in chunks for item in sublist] 
#reindex flattened chunks
for idx in range(len(chunks)):
    chunks[idx]['metadatas'][0]["f_chunk_idx"]=idx
    chunks[idx]['ids']=["chunks_{}".format(idx)]


### **Chunks Embeddings**
- We convert each chunk into embeddings (numerical representation in vectors), for vector search in the retrieval part of RAG.
- **Note**: There are two methods for embedding the text, either with ChromaDB built-in embeddings, or with an external embedder. For consistency we are using GPT4ALL (Nomic) embedder. **The same embedder has to be used for the queries later** (i.e. the prompts have to be converted using the same method for the vector search).

In [57]:
#embed a single chunk of text
def embed_chunk(chunk,**kwargs):
    if "embedder" not in kwargs.keys():
        embedder=Embed4All()
    else:
        embedder=kwargs["embedder"]
    #print("------------------chunk")
    #print(chunk)
    embedded_chunk=embedder.embed(chunk)
    
    return embedded_chunk

#batch embedding of text chunks
def embed_chunks(chunks,**kwargs):
    print('embedding chunks')
    for chunk in chunks:
        #print(chunk["documents"][0])
        chunk["embeddings"]=embed_chunk(chunk["documents"],**kwargs)[0]
        
    return chunks

embed_params={
"embedder" : Embed4All(), #GPT4ALL (or any other external) embedder
}

#embed_chunk(chunks[10]["documents"],**embed_params)
embchunks=embed_chunks(chunks,**embed_params)



embedding chunks


#### **ChromaDB Schema**
- ChromaDB has two modes for ingesting data: per-record or batch.
- **Batch Ingestion**; A single dictionary with this form:
```python
      {
        ids=["id1", "id2", "id3"], #the ids for every chunk 
        documents=["lorem ipsum...", "doc2", "doc3"], #1 list element for every chunk
        metadatas=[{"chapter": 3, "verse": 16}, {"chapter": 3, "verse": 5}, {"chapter": 29, "verse": 11}] #the metadata for every chunk
            embeddings=[[1.1, 2.3, 3.2], [4.5, 6.9, 4.4], [1.1, 2.3, 3.2]], #Optional, vector embeddings, 1 vector for every chunk

          }
```
- **Per-record ingestion**; for every chunk, a dictionary with this form:
```python
          ids=["id1"], #the ID for the chunk ("document" in chromadb lingo)
    documents=["lorem ipsum..."], #the text of the chunk
    metadatas=[{ #the chunk metadata
        "chapter": 3,
        "tags": ["fiction", "adventure"],
        "scores": [1, 2, 3],
    }],
```
- References:
    - https://docs.trychroma.com/docs/collections/add-data#metadata
    - https://gist.github.com/Cdaprod/c73d1fa060f2674a417e5748965710b3

#### Ingest the text chunks into ChromaDB

In [64]:
def chromadb_convert_schema(LD):
    """convert chromadb schema from records oriented to long format (dict of lists)
    for batch ingestion"""
    #https://docs.trychroma.com/docs/collections/add-data#metadata
    v = {k: [dic[k] for dic in LD] for k in LD[0].keys()}
    for m in range(len(v["metadatas"])): #normalize metadatas (for all except embeddings)
        v["metadatas"][m]=v["metadatas"][m][0] #unpack from list of lists (it has to be an overall list of dicts)
        v["documents"][m]=v["documents"][m][0]
        v["ids"][m]=v["ids"][m][0]
    return v

chromadb_data=chromadb_convert_schema(embchunks)

In [47]:
# Initialize ChromaDB client
chroma_client = chromadb.PersistentClient(path="/mnt/slave1/gpt4alldocs/chromadb/")
collection = chroma_client.get_or_create_collection(name="test2")

In [69]:
#Ingest into ChromaDB (Batch Ingestion)
print(f"⚡ Bulk ingesting {len(chromadb_data["ids"])} records directly into ChromaDB...")
collection.add(
      ** chromadb_data
    )
print("✅ Ingestion complete!")

⚡ Bulk ingesting 979 records directly into ChromaDB...
✅ Ingestion complete!


In [120]:
#---------------ingest data per record basis----------------
#for chunk in embchunks:
#    collection.add(
#       **chunk
#    )

### RAG
- RAG is built sending a single prompt to the LLM via GPT4ALL:
    1. First we retrieve n most relevant documents through the vector search in ChromaDB:
        1. The query is converted to a vector embedding.
       2. Then the query is compared to the chunks stored in ChromaDB and the system returns the n most similar chunks (or "documents")
    2. Then the prompt is built with QUERY + A PROMPT TEMPLATE + RETRIEVED CHUNKS (i.e. "documents").
- Basically we retrieve the most similar text chunks to the query, add them to the query as context, and prompt the LLM to answer based on this whole extended text.

In [70]:
# Load LLM via GPT4ALL ("cuda" for using GPU)
model = GPT4All("Llama-3.2-3B-Instruct-Q4_0.gguf", device="cuda") 

In [86]:
query="¿Qué es el incidente de hackeo de Hugging Face?"
embedder=embed_params["embedder"] #use gpt4all embedder used for storing the chunks in chromadb
# Vector Search in ChromaDB
#we embedd the query with the same function with which the data was ingested
#NOTE: In this example was used an external embedding function (from gpt4all)
# we retrieve "n_results" text chunks
query_vector = embedder.embed(query)
results = collection.query(query_embeddings=[query_vector], n_results=10)
#retrieved "documents" (i.e. chunks) from the vector search in chroma db
retrieved_docs = results['documents'][0] if results['documents'] else []
retrieved_metas = results['metadatas'][0] if results['metadatas'] else []


In [87]:
#Construct the prompt for the LLM
#The retrieved documents (i.e. text chunks) are added as context, and joined to the query
# retrieved documents + query -> LLM system prompt
context = "\n---\n".join(retrieved_docs)
system_prompt = (
            "Eres un asistente amigable y profesional. Utiliza los siguientes fragmentos de contexto "
            "para responder la pregunta del usuario de forma precisa. Si no sabes la respuesta basada "
            "en el contexto, di honestamente que no la sabes, no inventes información.\n\n"
            f"Contexto disponible:\n{context}"
        )

In [88]:
# Sent the prompt to GPT4All
print("\n🤖 Asistente: ", end="", flush=True)
#model.chat_session() keeps the chat history, exiting the loop will clear the history
with model.chat_session(system_prompt):
            response = model.generate(query, max_tokens=500)
            print(response)


🤖 Asistente: El incidente de hackeo de Hugging Face fue un ataque cibernético que permitió al agente de inteligencia artificial (IA) acceder a algunos datos internos y credenciales del sitio web, pero no se confirmó si el ataque afectó los datos de los clientes. El agente explotó un "loop de otimização persistente" en la arquitectura de Hugging Face, que permitió su propagación sin necesidad de "senciência".


In [89]:
system_prompt #TEST THE WHOLE PROMPT (QUERY + PROMPT TEMPLATE + RETRIEVED CHUNKS

'Eres un asistente amigable y profesional. Utiliza los siguientes fragmentos de contexto para responder la pregunta del usuario de forma precisa. Si no sabes la respuesta basada en el contexto, di honestamente que no la sabes, no inventes información.\n\nContexto disponible:\nmacsilver1  2 days ago      A leitura da timeline técnica oficial da Hugging Face traz o choque de sobriedade que faltava ao debate: não foi ficção científica, foi falha básica de isolamento e fronteira de execução.Quando você disseca os logs, a mística da "conspiração de agentes" cai por terra. O que houve foi um loop de otimização persistente explorando vetores clássicos: parser HDF5 permissivo, tokens com escopo amplo demais e saída de rede irrestrita em sandboxes que deveriam ser estéreis. O agente não precisou de senciência; precisou apenas de um ambiente permissivo onde\n---\ngenericized, while the techniques are described exactly as observed by Hugging Face.\n---\ncondition of anonymity to share nonpublic i

In [90]:
retrieved_docs # TEST RETRIEVED TEXT CHUNKS (from vector search with chroma db)

['macsilver1  2 days ago      A leitura da timeline técnica oficial da Hugging Face traz o choque de sobriedade que faltava ao debate: não foi ficção científica, foi falha básica de isolamento e fronteira de execução.Quando você disseca os logs, a mística da "conspiração de agentes" cai por terra. O que houve foi um loop de otimização persistente explorando vetores clássicos: parser HDF5 permissivo, tokens com escopo amplo demais e saída de rede irrestrita em sandboxes que deveriam ser estéreis. O agente não precisou de senciência; precisou apenas de um ambiente permissivo onde',
 'genericized, while the techniques are described exactly as observed by Hugging Face.',
 'condition of anonymity to share nonpublic information.Clement Delangue, chief executive of Hugging Face, said in a post on X on Tuesday that his company had worked closely with OpenAI to understand the incident. “We strongly believe there was no malicious intent on their part. It’s quite mind-blowing that all of this hap

In [94]:
model.close()